# Lab 1: Honest clouds

**AI in Space Applications II · after Lesson 8 · core task about 35 minutes, about 60 minutes in all · fill in the blanks**

**Goal.** Train a small cloud model for VEGA-1 on real Sentinel-2 chips and score it twice: with a random split of chips, and with a split by region. Then set its send or discard line from stated costs.

**What you will have at the end:** a trained model, two scores you can explain, a decision line with its derivation, and a pass mark you committed before training. Capstone Phase 1 starts from this notebook.

**How to work.** Run the cells in order. Where you see `____`, replace it with your own code: the comment beside it says what goes there. A cell marked **Checkpoint** tells you what you should see. If you get stuck, open the **Stuck?** hint under the cell.

**In Colab.** Opened this from the course's *Open in Colab* link? First choose **File → Save a copy in Drive**, or your edits are lost when you close the tab. When the setup cell asks to connect Google Drive, allow it: the notebook keeps your models and results in the Drive folder `AI_in_Space_II_labs`, where later labs and sessions find them. The data files download by themselves.

**Data.** The VEGA-1 training set: 480 chips of 128 × 128 pixels in bands B02 to B08, cut from 120 patches of CloudSEN12+ (through Copernicus-Bench's Cloud-S2 subset, CC0). Contains modified Copernicus Sentinel data.

## 0. Setup

This cell imports the libraries, fixes the random seeds and prints the versions. Copy the printed versions into your repository's `requirements.txt`.

In [ ]:
import json
import os
import platform
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn

FAST = os.environ.get("VEGA1_FAST") == "1"        # set only by the course's own tests: a short run that checks the code
SEED = 0

KEEP_DIR = None                                   # your folder in Google Drive, when this runs in Colab
try:
    from google.colab import drive                # Colab only: Colab forgets files when a session ends, Drive does not
    drive.mount("/content/drive")
    KEEP_DIR = "/content/drive/MyDrive/AI_in_Space_II_labs"
    os.makedirs(KEEP_DIR, exist_ok=True)
    for name in os.listdir(KEEP_DIR):             # bring back what you saved in earlier labs and sessions
        if not os.path.exists(name) and os.path.isfile(os.path.join(KEEP_DIR, name)):
            shutil.copy2(os.path.join(KEEP_DIR, name), name)
    print("Your Drive folder:", KEEP_DIR, "| files found there:", ", ".join(sorted(os.listdir(KEEP_DIR))) or "none yet")
except ImportError:
    pass                                          # not in Colab: files stay beside the notebook
except Exception as e:
    print("Google Drive is not connected, so files last only for this session:", e)


def keep(*names):
    """Copy these files to your Drive folder, so the next session and the next lab find them."""
    if KEEP_DIR:
        for name in names:
            if os.path.exists(name):
                shutil.copy2(name, os.path.join(KEEP_DIR, name))
        print("Kept in your Drive folder:", ", ".join(names))


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| NumPy", np.__version__, "| device:", DEVICE)

## 1. Warm-up (optional, a few minutes)

Skip this if NumPy arrays feel familiar. Two blanks, to check that slicing and averaging do what you expect.

In [ ]:
image = np.arange(7 * 4 * 4, dtype=np.float32).reshape(7, 4, 4)      # a toy "image": 7 bands, 4 x 4 pixels
red = image[____]                    # band index 2 is B04, red, when the bands are B02 ... B08
mean_per_band = image.mean(axis=____)   # one mean per band: average over the two pixel axes
print("red band shape:", red.shape, "| one mean per band:", mean_per_band.shape)
assert red.shape == (4, 4) and mean_per_band.shape == (7,)

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

The chip is stored as (bands, height, width), with the bands in the order B02, B03, B04 … B08. Count from zero to find red (B04).

For the second line: you want one number per band, so average over every axis that is *not* the band axis. NumPy's `axis=` accepts a tuple of axes.

</details>

## 2. Load the data, and look at it

The file holds the stored top-of-atmosphere values, the label of every pixel, and the **region** and **patch** each chip comes from. Stored values become reflectance as *value × scale + offset*, with the scale and offset read from the file (Lesson 3).

**Checkpoint.** You should see 480 chips, 7 bands, and reflectance mostly between 0 and 1.

In [ ]:
DATA_DIR = os.environ.get("VEGA1_DATA_DIR", ".")
DATA_URL = "https://github.com/Astronify-SpaceTech/ai-in-space-ii-labs/releases/download/data-v1/"   # the course's data files


def get_file(name):
    """Find a course data file: beside the notebook, else download it from the course's link, else upload it in Colab."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    if DATA_URL:
        import urllib.request
        try:
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(DATA_URL + name, path + ".part")
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            print(f"download failed ({e}); choose the file you downloaded from the lesson page instead")
    try:
        from google.colab import files       # Colab only: choose the file you downloaded from the lesson page
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} beside this notebook (it is attached to the lab on the course page).")


d = np.load(get_file("vega1_train.npz"))
X = d["X"].astype(np.float32) * float(d["scale"]) + float(d["offset"])     # stored value x scale + offset = reflectance
y = d["y"].astype(np.int64)                      # 0 clear, 1 thick cloud, 2 thin cloud, 3 cloud shadow
region, patch = d["region"], d["patch"]
CLASSES = ["clear", "thick cloud", "thin cloud", "cloud shadow"]

clear_share = (y == 0).mean(axis=(1, 2))         # share of each chip that is usable ground
usable = clear_share >= 0.5                      # R2 counts chips with at least half their area usable
print(f"{len(X)} chips, {X.shape[1]} bands, {X.shape[2]} x {X.shape[3]} pixels, {len(np.unique(region))} regions, "
      f"{len(np.unique(patch))} patches")
print("reflectance: min %.2f, median %.2f, max %.2f" % (X.min(), np.median(X), X.max()))
print("class shares:", {c: round(float((y == i).mean()), 3) for i, c in enumerate(CLASSES)})
print(f"usable chips (at least half clear): {usable.mean():.0%}")

In [ ]:
import matplotlib.pyplot as plt

i = int(np.argsort(clear_share)[len(X) // 2])    # a chip with a middling amount of cloud
fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
rgb = np.clip(X[i][[2, 1, 0]].transpose(1, 2, 0) * 3.0, 0, 1)     # B04, B03, B02 as red, green, blue, brightened
ax[0].imshow(rgb)
ax[0].set_title(f"chip {i}, region {region[i]}")
ax[1].imshow(y[i], vmin=0, vmax=3, cmap="viridis")
ax[1].set_title("label: 0 clear ... 3 shadow")
for a in ax:
    a.axis("off")
plt.show()

## 3. Write your pass mark first

Requirement R1 is judged against a pass mark **you** set, before you train. Here the measure is the **usable-area error**: for each chip, the difference between the clear share your model predicts and the true clear share, averaged over the test chips, in percentage points. Lower is better. A model that always answers "half clear" scores the baseline printed below.

Choose a number you would defend to a review board, write it in the cell, and run it. The cell saves it with the time, so you can commit `PASS_MARKS.md` to your repository **now**.

In [ ]:
baseline_error = float(np.abs(0.5 - clear_share).mean() * 100)
print(f"A model that always answers 'half clear' has a usable-area error of {baseline_error:.1f} points.")

PASS_MARK_POINTS = ____      # your pass mark: the largest usable-area error, in points, you will accept on a held-out region
with open("PASS_MARKS.md", "w", encoding="utf-8") as f:
    f.write(f"# Pass marks, committed before training\n\n- R1: usable-area error on a held-out region at most "
            f"{PASS_MARK_POINTS} points. Written {time.strftime('%Y-%m-%d %H:%M:%S %Z')}.\n")
print("Saved PASS_MARKS.md: commit it before you run the next section.")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

This blank is a decision, not a calculation. Look at the baseline's error printed above: a model worth flying should beat it clearly. Choose your number of points, write one line in `PASS_MARKS.md` saying why, and commit it **before** you train. There's no single right value: what counts is that you set it first, and can defend it.

</details>

## 4. The model and the training loop (given)

A small U-Net, as in Lesson 5: an encoder that halves the picture twice, a decoder that brings it back, and skip connections that carry the fine detail across. Read it once; you will not need to change it.

In [ ]:
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallUNet(nn.Module):
    """Input: (batch, 7, H, W) reflectance. Output: (batch, 4, H, W) class scores. H and W must divide by 4."""

    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1, self.enc2, self.mid = Block(bands, a), Block(a, b), Block(b, c)
        self.dec2, self.dec1 = Block(c + b, b), Block(b + a, a)
        self.head = nn.Conv2d(a, classes, 1)

    def features(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return e1, e2, m

    def forward(self, x):
        e1, e2, m = self.features(x)
        d2 = self.dec2(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1))
        d1 = self.dec1(torch.cat([Fn.interpolate(d2, scale_factor=2.0, mode="nearest"), e1], dim=1))
        return self.head(d1)


def train_model(X_train, y_train, make_model=SmallUNet, epochs=None, seed=SEED):
    """Train a model on the given chips and return it. About a minute on a GPU, a few minutes on a CPU."""
    set_seed(seed)
    epochs = epochs or (1 if FAST else 6)
    model = make_model().to(DEVICE)
    model.mean.copy_(torch.tensor(X_train.mean(axis=(0, 2, 3))).view(1, -1, 1, 1))      # normalise with the training
    model.std.copy_(torch.tensor(X_train.std(axis=(0, 2, 3)) + 1e-6).view(1, -1, 1, 1))  # chips' own statistics
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(xt))
        total = 0.0
        for k in range(0, len(order), 16):
            idx = order[k:k + 16]
            xb, yb = xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if random.random() < 0.5:                     # a mirror flip: the same chip, seen another way
                xb, yb = xb.flip(-1), yb.flip(-1)
            logits = model(xb)
            step = yb.shape[-1] // logits.shape[-1]       # a model with a coarser output is scored on coarser labels
            loss = Fn.cross_entropy(logits, yb[:, ::step, ::step])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss) * len(idx)
        print(f"  epoch {epoch + 1}/{epochs}: loss {total / len(order):.3f} ({time.time() - t0:.0f} s)")
    return model.eval()


@torch.no_grad()
def predict(model, X_any, batch=32):
    """Class probabilities for every pixel, at the chip's own size: (chips, 4, H, W)."""
    out = []
    for k in range(0, len(X_any), batch):
        xb = torch.from_numpy(X_any[k:k + batch]).to(DEVICE)
        p = torch.softmax(model(xb), dim=1)
        if p.shape[-1] != xb.shape[-1]:
            p = Fn.interpolate(p, size=xb.shape[-2:], mode="nearest")
        out.append(p.cpu().numpy())
    return np.concatenate(out)

## 5. How good is a mask? (fill in)

Two scores. **Cloud IoU** (Lesson 5): overlap ÷ union of the predicted and the true cloud pixels, with thick and thin cloud counted together. **Usable-area error:** the measure your pass mark uses.

In [ ]:
def scores(model, X_test, y_test):
    prob = predict(model, X_test)
    pred = prob.argmax(axis=1)
    pred_cloud, true_cloud = np.isin(pred, (1, 2)), np.isin(y_test, (1, 2))
    overlap = (pred_cloud ____ true_cloud).sum()      # pixels that are cloud in both: & means "and"
    union = (pred_cloud ____ true_cloud).sum()        # pixels that are cloud in either: | means "or"
    iou = ____ / ____
    pred_clear_share = prob[:, 0].mean(axis=(1, 2))               # the model's clear share for each chip
    true_clear_share = (y_test == 0).mean(axis=(1, 2))
    area_error = np.abs(____ - ____).mean() * 100      # mean difference between predicted and true clear share, in points
    return {"cloud_iou": float(iou), "usable_area_error_points": float(area_error),
            "pixel_accuracy": float((pred == y_test).mean())}

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

Two arrays above hold one clear share per chip: one from the model, one from the labels. The error is how far apart they are, chip by chip, then averaged. Check the names defined a few lines up.

</details>

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

IoU is overlap ÷ union (Lesson 5). Both masks are arrays of True and False. Which operator keeps a pixel only when it's True in *both*? Which keeps it when it's True in *either*? The comments beside each line name them.

</details>

## 6. Core task, part 1: a random split of chips

The split every tutorial uses: shuffle the chips, keep a quarter for testing.

**Estimate first.** Before you run it: will this score be close to what the satellite achieves over a region it has never seen?

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(X))
n_test = len(X) // 4
test_random, train_random = order[:____], order[____:]      # the first quarter of the shuffled chips tests, the rest train
print(f"random split: {len(train_random)} chips train, {len(test_random)} test")
model_random = train_model(X[train_random], y[train_random])
score_random = scores(model_random, X[test_random], y[test_random])
print("random split:", {k: round(v, 3) for k, v in score_random.items()})

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

Which variable above says how many chips go to the test set? A slice `[:k]` takes the first k items of the shuffled order and `[k:]` takes the rest: the test set gets the start, training the remainder.

</details>

## 7. Core task, part 2: a split by region

Now hold out whole regions. No chip in the test set may share a region with a training chip.

**Checkpoint.** Compare the two scores, but notice that they come from different test chips, so they are not a like-for-like pair. Whatever the numbers, only this one describes ground the model has never seen: section 8 shows why. In Lesson 7's world the random split flattered the model badly; with a small network trained for a few epochs on real chips the gap can be small, or even go the other way. Report the honest number, against your pass mark.

In [ ]:
regions = np.random.default_rng(SEED).permutation(np.unique(region))      # a fixed, seeded draw of regions
test_regions = regions[:max(1, int(len(regions) * 0.25))]                  # a quarter of the regions are held out
is_test = np.isin(____, ____)                          # True for chips whose region is held out
test_region_idx, train_region_idx = np.where(____)[0], np.where(____)[0]
print(f"region split: {len(train_region_idx)} chips train, {len(test_region_idx)} test, held-out regions {sorted(test_regions.tolist())}")
model = train_model(X[train_region_idx], y[train_region_idx])
score_region = scores(model, X[test_region_idx], y[test_region_idx])
print("region split:", {k: round(v, 3) for k, v in score_region.items()})
print(f"\nR1: usable-area error {score_region['usable_area_error_points']:.1f} points on held-out regions; "
      f"my pass mark was {PASS_MARK_POINTS} -> {'PASS' if score_region['usable_area_error_points'] <= PASS_MARK_POINTS else 'MISS'}")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

`np.isin(a, b)` asks, for each element of `a`, "is it one of `b`?". Which array gives each chip's region, and which lists the regions held out?

On the second line, `np.where(mask)[0]` gives the positions where a mask is True. The training chips are where it's False: NumPy's `~` flips a True/False array.

</details>

## 8. Check, then break

Engineers check a tool against a case they understand before they trust it. The check: **no region is on both sides of the split.** Run it on your region split (it should pass), then on the random split (it should fail, and tell you how badly).

In [ ]:
def check_split(train_idx, test_idx, groups, name):
    shared = set(groups[train_idx]) & set(groups[test_idx])
    leaked = np.isin(groups[test_idx], list(shared)).mean()
    print(f"{name}: {len(shared)} groups on both sides; {leaked:.0%} of test chips have a group-mate in training")
    return len(shared) == 0


assert check_split(train_region_idx, test_region_idx, region, "region split, by region")
ok = check_split(____, ____, region, "random split, by region")     # run the same check on the random split
check_split(____, ____, patch, "random split, by patch")            # and by patch: chips cut from the same picture
print("The random split", "passes" if ok else "FAILS the check: its score describes regions the model has already seen.")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

This is the same check you just ran, on a different split. Which two index arrays did section 4 make for the random split? Pass them in the order the function expects (look at its first two arguments).

</details>

## 9. Your turn: the line between sending and discarding

VEGA-1 sends a chip when the model's **probability that the chip is usable** is above a line. Lesson 8 derived where the line goes: the expected cost of sending, (1 − p) × C_S, equals the expected cost of discarding, p × C_D, at p* = C_S ÷ (C_S + C_D).

1. **Calibrate.** The model's raw clear share is a score, not yet a probability. Fit a calibration on chips the model has not trained on, then check it with a reliability table.
2. **State your costs** and compute the line.
3. **Count** the usable chips your line discards on the held-out regions. R2 allows at most 2%.

In [ ]:
from sklearn.linear_model import LogisticRegression

# hold back some training regions for calibration (the model above did train on them, so for the capstone, retrain without them)
cal_regions = np.unique(region[train_region_idx])[::4]
cal_idx = train_region_idx[np.isin(region[train_region_idx], cal_regions)]


def chip_score(model, X_any):
    return predict(model, X_any)[:, 0].mean(axis=(1, 2))           # the model's clear share: a score between 0 and 1


def to_logit(s):
    s = np.clip(s, 1e-4, 1 - 1e-4)
    return np.log(s / (1 - s)).reshape(-1, 1)


platt = LogisticRegression().fit(to_logit(chip_score(model, X[cal_idx])), usable[cal_idx])
p_usable = platt.predict_proba(to_logit(chip_score(model, X[test_region_idx])))[:, 1]
truth = usable[test_region_idx]


def reliability(p, truth, bins=5):
    """Mean predicted probability against the share that really was usable, bin by bin, and the calibration error."""
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if m.any():
            print(f"  said {p[m].mean():.2f}  was {truth[m].mean():.2f}  ({m.sum()} chips)")
            ece += m.mean() * abs(p[m].mean() - truth[m].mean())
    return ece


print("Reliability on the held-out regions (said: mean probability; was: share really usable):")
print(f"calibration error: {reliability(p_usable, truth):.3f}")

In [ ]:
C_S, C_D = ____, ____                   # your costs: a wrong send, and a wrong discard, in the same unit
p_star = ____ / (____ + ____)           # Lesson 8's line
send = p_usable > p_star
false_discard = (truth & ____).sum() / max(truth.sum(), 1)      # usable chips that were NOT sent, as a share of usable chips
print(f"line p* = {p_star:.3f}; chips discarded: {(~send).mean():.0%} of the downlink saved")
print(f"R2: usable chips discarded = {false_discard:.1%}")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

A usable chip is wrongly discarded when it is usable **and** not sent. `send` is True for chips that go down. How do you write "not sent" for an array of True and False?

</details>

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

The two costs are your judgement, in the same unit (only their ratio matters). What does a wrong *send* waste? What does a wrong *discard* lose?

For the line, go back to Lesson 8: write the expected cost of sending and of discarding a tile whose calibrated score is p, set them equal, and solve for p.

</details>

**Checkpoint.** If a wrong discard costs more than a wrong send, your line is below one half. If your false-discard share is above 2%, do not nudge the line until it passes: ask whether the model, the calibration or your costs need to change, and write down which.

## 10. Save your work

This cell saves the model and a small results file, and logs the run with MLflow in a SQLite store (a plain folder does not work with this version). Then commit the notebook, `PASS_MARKS.md` and `lab1_results.json` to your repository.

In [ ]:
torch.save({"state": model.state_dict(), "widths": (16, 32, 64), "seed": SEED,
            "test_regions": sorted(int(r) for r in test_regions)}, "lab1_model.pt")
results = {"pass_mark_points": PASS_MARK_POINTS, "random_split": score_random, "region_split": score_region,
           "costs": {"C_S": C_S, "C_D": C_D}, "p_star": float(p_star), "false_discard": float(false_discard),
           "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__}}
with open("lab1_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=1)

import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("vega1-lab1")
with mlflow.start_run(run_name="region-split"):
    mlflow.log_params({"seed": SEED, "split": "by region", "C_S": C_S, "C_D": C_D})
    mlflow.log_metrics({"cloud_iou": score_region["cloud_iou"], "usable_area_error_points": score_region["usable_area_error_points"],
                        "false_discard": float(false_discard)})
print("Saved lab1_model.pt and lab1_results.json; logged one run to mlflow.db")
keep("lab1_model.pt", "lab1_results.json", "mlflow.db")
print(json.dumps({k: results[k] for k in ("pass_mark_points", "random_split", "region_split", "p_star", "false_discard")}, indent=1))

## What you have now

- A pass mark with a time on it, older than your training run.
- Two scores for one model design, and a reason why they differ.
- A line derived from costs, and the share of usable chips it discards.

**For Capstone Phase 1:** retrain with the calibration regions kept out of training, choose your own costs, and write two sentences on what you would try next if you missed your pass mark.

## Stretch (optional, not graded)

- **Your own region.** Search Sentinel-2 scenes through Earth Search's public catalogue with `pystac-client` and read a window of one band with `rasterio`. No account is needed. Remember Lesson 3: read the scale and offset from the metadata before you compare scenes.
- **Flood tiles.** STURM-Flood has Sentinel-2 flood tiles with water masks (CC BY 4.0). Could your model's clear share help find them, or would you need a second model?
- **One more split.** Hold out a different quarter of the regions. How much does your honest score move? That spread is part of your evidence.

## Hand in

1. Make sure every cell has run, top to bottom, with its outputs showing.
2. **File → Download → Download .ipynb**, and upload that file on the lab's page, with `lab1_results.json` from your Drive folder `AI_in_Space_II_labs`.
3. In the notes box, say which AI tools you used, if any.

Submitting marks the lab complete, so you can go on to the next lesson while it's graded. How it's graded is in the lab's instructions.